# 🚗 TOYOTA EV LifePilot — 油轉電 AI 隨行顧問（整合版）

把兩份筆記本合成一支程式：

| 功能 | 來源 |
|---|---|
| 車主檔案（稱呼／車款／里程／用車習慣）＋ 智能對話（純文字回答，不畫圖，速度較快） | `Toyota.ipynb` |
| CDP 資料載入 ＋ 5 年 TCO 精算儀表板 ＋ EV LifePilot 顧問人設 | `油車電車.ipynb` |
| 整合：AI 回答時會一併參考「車主檔案 + 已精算的 TCO + CDP 資料」 | 本筆記本新增 |

**使用前準備**
1. Colab 左側 🔑「密鑰」新增名稱為 `api` 的 Secret（值是 Gemini API Key），並開啟「筆記本存取權」。**請勿把 key 寫進程式碼。**
2. 把 CDP 的兩份 CSV 上傳到 Colab 檔案區（沒上傳也能執行，只是 AI 不會引用 CDP 資料）：
   - `CDP資料.csv`（或 `2026 AI黑客松CDP資料 (T車輛)`）
   - `2026 AI黑客松CDP 電動車主補充資料資料 (T車輛)`
3. 由上而下依序執行所有 cell；最後一格會產生 `gradio.live` 連結。

In [ ]:
!pip install -q -U google-genai gradio
!pip install -q plotly pandas

In [ ]:
# ---------------------------------------------------------------------------------
# 【區塊 0：套件匯入與 Gemini 初始化】
# ---------------------------------------------------------------------------------
import os
import re
import time
from pathlib import Path

import gradio as gr
import pandas as pd
import plotly.graph_objects as go
from google import genai
from google.genai import errors as genai_errors
from google.genai import types

try:
    import httpx
    TRANSPORT_ERRORS = (httpx.TransportError,)   # 斷線、逾時等網路層錯誤（google-genai 底層用 httpx）
except ImportError:
    TRANSPORT_ERRORS = ()

MODEL = "gemini-3.8-flash"   # 換成你帳號可用的 Gemini 模型名稱
MAX_RETRIES = 4              # 遇到 503（模型忙碌）/ 429（額度）/ 斷線 時最多嘗試幾次
REQUEST_TIMEOUT_MS = 45_000  # 單次請求逾時；避免連線卡住時整個介面一直轉圈


def load_api_key():
    """優先讀 Colab Secrets 的 `api`，讀不到再讀環境變數；key 不要寫死在程式碼裡。"""
    try:
        from google.colab import userdata
        return userdata.get("api")
    except Exception:
        return os.environ.get("GEMINI_API_KEY") or os.environ.get("GOOGLE_API_KEY")


api_key = load_api_key()
client = (genai.Client(api_key=api_key, http_options=types.HttpOptions(timeout=REQUEST_TIMEOUT_MS))
          if api_key else None)
if client is None:
    print("⚠️ 找不到 API Key：請在 Colab 左側「密鑰」新增名稱為 api 的 Secret，並開啟筆記本存取權。")
else:
    print("✅ Gemini 用戶端已就緒")

## 📦 區塊 1：載入 CDP 資料
自動在目前資料夾與 `/content` 找檔案（檔名有沒有 `.csv` 都行；編碼依序嘗試 utf-8 → big5 → cp950 → big5hkscs）。找不到不會中斷，只是 AI 不會引用該資料；上傳後重新執行本 cell，AI 下一則回覆就會用到。
也可以直接在下方填 `CDP_PATH` / `EV_PATH` 指定路徑。

> 🔒 **個資**：預設只把「欄位統計」（平均、分布等，不含任何一位客戶的逐列資料）交給 AI。
> 若把 `SEND_ROW_LEVEL_DATA` 改成 `True`，整張表的逐列資料會送給 Gemini；而 `share=True` 的公開連結任何人都能問到這些內容，請先確認可以公開。

In [ ]:
# ---------------------------------------------------------------------------------
# 【區塊 1：CDP 資料載入與車主查詢】
# ---------------------------------------------------------------------------------
CDP_PATH = None   # 主 CDP 表（含「編號」「現保有車款_T」「居住縣市」...）；None = 自動尋找
EV_PATH = None    # 電動車主補充資料；None = 自動尋找

# False（預設）= 只給 AI 欄位統計；True = 逐列資料也給（含客戶個資，公開連結下任何人都問得到）
SEND_ROW_LEVEL_DATA = False

ENCODINGS = ("utf-8-sig", "big5", "cp950", "big5hkscs")


def find_data_file(include, exclude=()):
    """在目前資料夾與 /content 找檔名含所有 include 關鍵字的檔案（.csv 或無副檔名；優先 .csv，其次最近修改的）。"""
    for folder in (Path.cwd(), Path("/content")):
        if not folder.is_dir():
            continue
        hits = [p for p in folder.iterdir()
                if p.is_file() and not p.name.startswith((".", "~"))
                and p.suffix.lower() in (".csv", "")
                and all(k in p.name for k in include)
                and not any(k in p.name for k in exclude)]
        hits.sort(key=lambda p: (p.suffix.lower() != ".csv", -p.stat().st_mtime))
        if hits:
            return hits[0]
    return None


def read_csv_any(path):
    """依序嘗試多種編碼讀 CSV。"""
    last_error = None
    for enc in ENCODINGS:
        try:
            return pd.read_csv(path, encoding=enc)
        except UnicodeDecodeError as e:
            last_error = e
    raise last_error


def clean_columns(columns):
    """欄名去掉換行/空白（例如「近一年回廠有費定保次數\\n(定期保養)」）；去除後撞名的加上 _2、_3。"""
    seen, cleaned = {}, []
    for c in columns:
        name = re.sub(r"\s+", "", str(c)) or "欄位"
        seen[name] = seen.get(name, 0) + 1
        cleaned.append(name if seen[name] == 1 else f"{name}_{seen[name]}")
    return cleaned


def load_table(label, explicit_path, include, exclude=()):
    """載入一份 CDP 表；失敗時回傳 None 而不是丟例外。"""
    try:
        path = Path(explicit_path) if explicit_path else find_data_file(include, exclude)
        if path is None or not path.is_file():
            print(f"⚠️ 找不到「{label}」{f'（{explicit_path}）' if explicit_path else ''}：AI 回答將不引用這份資料"
                  "（上傳 CSV 後重新執行本 cell 即可）")
            return None
        df = read_csv_any(path)
    except Exception as e:
        print(f"⚠️ 「{label}」讀取失敗：{e}")
        return None
    df.columns = clean_columns(df.columns)
    print(f"✅ 已載入「{label}」：{path.name}（{len(df)} 列 × {df.shape[1]} 欄）")
    return df


def find_col(df, keyword):
    """先找完全相同的欄名，再找「欄名包含 keyword」的第一欄。"""
    if keyword in df.columns:
        return keyword
    return next((c for c in df.columns if keyword in c), None)


df_cdp = load_table("CDP 主資料表", CDP_PATH, include=("CDP資料",), exclude=("補充",))
df_ev = load_table("電動車主補充資料", EV_PATH, include=("電動車主補充",))
if df_cdp is not None and find_col(df_cdp, "編號") is None:
    print("⚠️ CDP 主資料表找不到「編號」欄，請確認檔案是否正確（分隔符號或編碼不對時，常會讀成只有 1 欄）")


def cut_lines(text, limit):
    """依「整行」截斷到 limit 字元以內，避免資料在行中間被切斷。"""
    if len(text) <= limit:
        return text
    kept, used = [], 0
    for line in text.splitlines():
        if used + len(line) + 1 > limit:
            break
        kept.append(line)
        used += len(line) + 1
    return "\n".join(kept)


def summarize_table(df, max_chars):
    """只給欄位與統計（不含任何一列的逐列資料）：數值欄的 describe、類別欄（≤15 種）的次數分布。"""
    parts = [f"（資料共 {len(df)} 列 × {df.shape[1]} 欄；以下只有欄位統計，不含逐列資料）",
             "[欄位] " + "、".join(map(str, df.columns))]
    try:
        numeric = df.select_dtypes("number")
        if not numeric.empty:
            parts += ["[數值欄統計]", numeric.describe().T.to_string()]
        for i, col in enumerate(df.columns):
            s = df.iloc[:, i]
            if pd.api.types.is_numeric_dtype(s) and not pd.api.types.is_bool_dtype(s):
                continue
            if 0 < s.nunique(dropna=True) <= 15:
                parts += [f"[{col} 次數分布]", s.value_counts(dropna=True).to_string()]
    except Exception as e:   # 統計失敗不該讓整個 cell 中斷
        parts.append(f"（部分統計無法產生：{e}）")
    return cut_lines("\n".join(parts), max_chars)


def table_for_prompt(df, max_chars=20000):
    """把資料表轉成給 AI 的文字（預設只有統計；SEND_ROW_LEVEL_DATA=True 才附逐列資料，且有長度上限）。"""
    if df is None:
        return "（未載入）"
    if not SEND_ROW_LEVEL_DATA:
        return summarize_table(df, max_chars)
    if len(df) <= 2000:
        full = df.to_string(index=False)
        if len(full) <= max_chars:
            return full
    head = cut_lines(df.head(30).to_string(index=False), max_chars // 2)
    return summarize_table(df, max_chars // 2) + f"\n[前 30 列（可能因過長而截短）]\n{head}"


def format_value(v):
    """None = 該欄沒有值；整數型的小數（3.0）顯示成 3。"""
    if pd.isna(v):
        return None
    if isinstance(v, float) and v.is_integer():
        return str(int(v))
    return str(v)


def load_owner(owner_id):
    """【函數功能】依車主編號從 CDP 主資料表撈出車款／居住縣市／定保次數，回傳 (狀態訊息, 車主資料文字)"""
    try:
        oid = int(owner_id)
    except (TypeError, ValueError, OverflowError):
        oid = 0
    if oid <= 0:   # 編號欄位預設會顯示 0，視同沒填
        return "⚠️ 請先輸入車主編號", ""
    if df_cdp is None:
        return "⚠️ 尚未載入 CDP 主資料表（請上傳 CSV 後重新執行資料載入 cell）", ""
    id_col = find_col(df_cdp, "編號")
    if id_col is None:
        return "⚠️ CDP 資料表找不到「編號」欄", ""
    rows = df_cdp[pd.to_numeric(df_cdp[id_col], errors="coerce") == oid]
    if rows.empty:
        return f"⚠️ CDP 資料表中沒有編號 {oid} 的車主", ""
    row = rows.iloc[0]
    fields = {"現保有車款": find_col(df_cdp, "現保有車款"),
              "居住縣市": find_col(df_cdp, "居住縣市"),
              "近一年有費定期保養次數": find_col(df_cdp, "定保次數")}
    lines = [f"- 車主編號：{oid}"]
    for label, col in fields.items():
        value = format_value(row[col]) if col is not None else None
        if value is not None:
            lines.append(f"- {label}：{value}")
    note = f"（共 {len(rows)} 筆相同編號，僅帶入第一筆）" if len(rows) > 1 else ""
    return f"✅ 已帶入編號 {oid} 的 CDP 資料{note}", "\n".join(lines)

## 💰 區塊 2：5 年 TCO（持有成本）精算與儀表板
下列費率都是**假設值**，會顯示在診斷報告上並一併提供給 AI 引用；想用 CDP 實際中位數校準，直接改 `TCO_ASSUMPTIONS` 即可。

In [ ]:
# ---------------------------------------------------------------------------------
# 【區塊 2：5 年 TCO (Total Cost of Ownership) 養車費用預估計算邏輯】
# ---------------------------------------------------------------------------------
TCO_ASSUMPTIONS = {
    "gas_price_per_l": 31.0,        # 95 無鉛汽油平均油價 (NT$/公升)
    "ice_fuel_eff": 12.0,           # 油車平均綜合油耗 (公里/公升)
    "elec_price_per_kwh": 4.5,      # 純電車平均電價 (NT$/度，採家用與公充混合加權)
    "ev_efficiency": 5.8,           # TOYOTA bZ4X 平均電耗 (公里/度)
    "ice_maint_per_year": 12000,    # 油車年均保養維修花費 (NT$/年)
    "ev_maint_per_year": 4500,      # bZ4X 年均保養維修花費 (NT$/年，無機油/火星塞等消耗品)
}


def assumptions_text():
    a = TCO_ASSUMPTIONS
    return (f"油價 {a['gas_price_per_l']:g} 元/公升、油車油耗 {a['ice_fuel_eff']:g} 公里/公升、"
            f"電價 {a['elec_price_per_kwh']:g} 元/度、bZ4X 電耗 {a['ev_efficiency']:g} 公里/度、"
            f"年保養 油車 {a['ice_maint_per_year']:,} 元／bZ4X {a['ev_maint_per_year']:,} 元")


def calculate_tco(current_car_type, annual_km, years=5):
    """
    【函數功能】計算 N 年養車與能源總費用對比 (燃油車 vs. TOYOTA bZ4X)
    【輸入參數】
      - current_car_type (str): 車主目前駕駛的油車車型（目前所有車型共用同一組油耗假設，保留參數以便日後分車型調校）
      - annual_km (int): 車主預估的年行駛里程數 (單位: 公里)
      - years (int): 比較年限，預設為 5 年
    【輸出結果】
      - total_ice (float): 油車總估算費用 (能源 + 保養)
      - total_ev (float): bZ4X 總估算費用 (能源 + 保養)
      - saved_money (float): 換購 bZ4X 預估可省下的總金額
    """
    a = TCO_ASSUMPTIONS
    ice_energy_cost = (annual_km / a["ice_fuel_eff"]) * a["gas_price_per_l"] * years
    ev_energy_cost = (annual_km / a["ev_efficiency"]) * a["elec_price_per_kwh"] * years
    total_ice = ice_energy_cost + a["ice_maint_per_year"] * years
    total_ev = ev_energy_cost + a["ev_maint_per_year"] * years
    return total_ice, total_ev, total_ice - total_ev


def update_dashboard(car_type, annual_km):
    """【函數功能】重新計算數據並產出視覺化圖表與診斷報告文字"""
    car_type = car_type or "目前車款"
    annual_km = int(annual_km)
    ice_cost, ev_cost, saved = calculate_tco(car_type, annual_km)

    fig = go.Figure(data=[
        go.Bar(name="現有燃油車", x=["5年總花費(能源+保養)"], y=[ice_cost], marker_color="#E53935"),  # 紅色代表油車
        go.Bar(name="TOYOTA bZ4X", x=["5年總花費(能源+保養)"], y=[ev_cost], marker_color="#4CAF50"),  # 綠色代表純電車
    ])
    verb = "省下" if saved >= 0 else "多花"
    fig.update_layout(
        title=f"<b>換購 bZ4X 5 年預估可為您{verb}<br>NT$ {abs(saved):,.0f} 元</b>",   # 換行：手機窄螢幕才不會被截斷
        barmode="group", height=380, yaxis_title="NT$", template="plotly_white",
        margin=dict(l=20, r=20, t=80, b=20),
    )
    summary = (f"💡 **專屬診斷報告**：依您駕駛 **{car_type}**、每年行駛 **{annual_km:,} 公里** 計算，"
               f"換購 bZ4X 5 年累積可{verb}約 **NT$ {abs(saved):,.0f} 元** 的油資與保養總支出！\n\n"
               f"> 以上為估算，假設：{assumptions_text()}。實際金額以原廠報價與個人用車情形為準。")
    return fig, summary

## 🤖 區塊 3：Gemini 隨行顧問（人設 + 資料庫 + 純文字回答）
- 人設（EV LifePilot、白話比喻、同理心）來自 `油車電車.ipynb`
- 檔案式對話（車主檔案 + 歷史紀錄）來自 `Toyota.ipynb`；AI 回覆一律是純文字，**不再產生圖表**，所以回得比較快
- 5 年 TCO 圖仍在上方儀表板，由程式直接計算，不經過 AI
- 遇到 503（模型忙碌）／429（額度）會自動退避重試，不再直接顯示錯誤

In [ ]:
# ---------------------------------------------------------------------------------
# 【區塊 3：Gemini API 隨行顧問 — System Instruction、純文字回答、對話邏輯】
# ---------------------------------------------------------------------------------
PERSONA = """
你是一位 TOYOTA 專屬的「EV LifePilot 油轉電 AI 隨行顧問」，使用繁體中文，語氣親切、專業、有同理心。
你的使命是幫助燃油車主消除對電動車 (bZ4X) 的疑慮（如充電焦慮、電池壽命、社區大樓裝樁、養車成本），
並協助評估油轉電後的五年持有成本差異。
請用白話的比喻回答（例如：將 150kW 快充比喻為喝一杯咖啡的時間；將 8 年保固比喻為繞台灣 130 圈）。
上面的數字只是比喻示例，實際的保固、充電功率、價格一律以原廠公告與下方資料為準。
"""

ANSWER_RULES = """
請直接用一般文字回答（可用 Markdown 條列），不要輸出 JSON、程式碼區塊或圖表。
規則：
1. 請嚴格根據【車主檔案】、【系統已精算的成本估算】與【資料庫】回答與計算；資料庫沒有涵蓋的部分，請說明是依一般行情估算。
2. 涉及費用、油耗/電耗等數字時，請說明假設（如年里程、油電價），並提醒實際以原廠報價為準。
3. 不可編造原廠優惠、保固條款或確切價格；不確定就說明並給出範圍。
"""


def build_system_instruction():
    return (f"{PERSONA}\n{ANSWER_RULES}\n"
            f"\n【資料庫 1：CDP 主資料表（車款、居住縣市、定期保養次數等）】\n{table_for_prompt(df_cdp)}\n"
            f"\n【資料庫 2：電動車主補充資料（保養費用、用車時數等）】\n{table_for_prompt(df_ev)}\n")


_SI_CACHE = {}


def get_system_instruction():
    """依「目前」載入的 CDP 資料組出系統指令；資料沒變就用快取，重跑資料 cell 後會自動更新。"""
    key = (id(df_cdp), id(df_ev), SEND_ROW_LEVEL_DATA)
    if _SI_CACHE.get("key") != key:
        _SI_CACHE.update(key=key, hold=(df_cdp, df_ev), text=build_system_instruction())   # hold：避免 id 被重複使用
    return _SI_CACHE["text"]


RETRYABLE_CODES = {408, 429, 500, 502, 503, 504}


def generate_text(prompt):
    """呼叫 Gemini；遇到暫時性錯誤（503/429/斷線...）以 2、4、8 秒退避重試。"""
    if client is None:
        raise RuntimeError("尚未設定 API Key：請在 Colab「密鑰」新增名稱為 api 的 Secret 後重新執行。")
    config = types.GenerateContentConfig(system_instruction=get_system_instruction())
    for attempt in range(MAX_RETRIES):
        try:
            return client.models.generate_content(model=MODEL, contents=prompt, config=config).text
        except (genai_errors.APIError, *TRANSPORT_ERRORS) as e:
            retryable = isinstance(e, TRANSPORT_ERRORS) or getattr(e, "code", None) in RETRYABLE_CODES
            if retryable and attempt < MAX_RETRIES - 1:
                time.sleep(2 ** (attempt + 1))
                continue
            raise


def friendly_error(e):
    code = getattr(e, "code", None)
    if isinstance(e, RuntimeError):   # 例如尚未設定 API Key：訊息本身就是指示
        return f"⚠️ {e}"
    if code == 503:
        return "AI 顧問目前較忙碌（503），已自動重試仍未成功，請稍後再送一次。"
    if code == 429:
        return "AI 請求次數已達上限（429），請稍等一分鐘再試，或檢查 API 額度。"
    if TRANSPORT_ERRORS and isinstance(e, TRANSPORT_ERRORS):
        return "連線到 AI 服務失敗或逾時，已自動重試仍未成功，請稍後再送一次。"
    return f"伺服器錯誤，請稍後再試。({str(e)[:200]})"


def _text(content):
    """Gradio 的訊息內容可能是字串、區塊清單或其他元件；只取文字部分。"""
    if isinstance(content, str):
        return content
    if isinstance(content, list):
        return "".join(c.get("text") or "" if isinstance(c, dict) else (c if isinstance(c, str) else "") for c in content)
    if isinstance(content, dict):
        return content.get("text") or ""
    return ""


def build_chat_prompt(history, user_message, name, car, mileage, annual_km, pref, cdp_info):
    lines = ["【車主檔案】",
             f"- 車主姓名：{name}",
             f"- 目前駕駛車款：{car}",
             f"- 目前里程數：{mileage}",
             f"- 預估年行駛里程：{annual_km} 公里",
             f"- 用車習慣/喜好：{pref}"]
    if cdp_info:
        lines += ["", "【CDP 車主資料（供交叉比對；若與車主檔案的車款不同，以車主檔案為準）】", cdp_info]
    try:
        ice, ev, saved = calculate_tco(car, annual_km)
        lines += ["", "【系統已精算的 5 年持有成本估算】",
                  f"- 油車：約 NT$ {ice:,.0f}", f"- bZ4X：約 NT$ {ev:,.0f}", f"- 差額（油車 − bZ4X）：約 NT$ {saved:,.0f}",
                  f"- 估算假設：{assumptions_text()}"]
    except Exception:
        pass   # 年里程沒填好時，就不附 TCO
    lines += ["", "--- 歷史對話紀錄 ---"]
    for m in history[-20:]:   # 只帶最近 20 則，避免對話越長越慢
        text = _text(m.get("content")) if isinstance(m, dict) else ""
        if text:
            lines.append(f"{'車主' if m.get('role') == 'user' else '顧問'}：{text}")
    lines += [f"車主：{user_message}", "顧問："]
    return "\n".join(lines)


def chat_with_ai(user_message, history, name, car, mileage, annual_km, pref, cdp_info):
    """產生器：先立刻顯示車主的提問並清空輸入框，等 AI 回覆後再補上答案。"""
    history = history or []
    if not user_message or not user_message.strip():
        yield "", history
        return

    prompt = build_chat_prompt(history, user_message, name, car, mileage, annual_km, pref, cdp_info)
    history = history + [{"role": "user", "content": user_message}]
    yield "", history
    history = list(history)   # 之後的 append 只動副本，不去改已經送出去的那一份
    try:
        answer = (generate_text(prompt) or "").strip()
        history.append({"role": "assistant", "content": answer or "（沒有取得回覆）"})
    except Exception as e:
        history.append({"role": "assistant", "content": friendly_error(e)})
    yield "", history

## 🖥️ 區塊 4：Gradio 介面並啟動
上方：左欄建立車主檔案、右欄顯示 5 年 TCO 圖與報告；下方：AI 隨行顧問對話（建立檔案後才會開放輸入）。

In [ ]:
# ---------------------------------------------------------------------------------
# 【區塊 4：建立 Gradio Web UI 介面並啟動分享連結】
# ---------------------------------------------------------------------------------
CAR_CHOICES = ["ALTIS", "RAV4", "CAMRY", "COROLLA CROSS", "其他油車"]


def confirm_profile(name, car, mileage, annual_km, pref):
    if not name or not car:
        return "⚠️ 請至少填寫您的稱呼與車款！", gr.update(interactive=False)
    return (f"✅ 檔案建立成功！您好，{name}。\n系統已紀錄您的愛車為 {car}，請在下方開始對話。",
            gr.update(interactive=True, placeholder="例如：大樓沒辦法裝充電樁怎麼辦？ bZ4X 電池保固多久？（Enter 送出）"))


def build_app():
    with gr.Blocks(title="EV LifePilot - TOYOTA 油轉電 AI 隨行顧問") as demo:
        gr.Markdown("# 🚗 TOYOTA EV LifePilot — 個人化油轉電 AI 隨行顧問")
        gr.Markdown("先建立您的車主檔案，AI 會為您精算 5 年持有成本，並依檔案客製化回答油轉電疑慮。")

        # 第一排：左欄車主檔案 / 右欄 TCO 儀表板
        with gr.Row():
            with gr.Column(scale=1):
                gr.Markdown("### 📋 步驟 1：建立車主檔案")
                owner_id_input = gr.Number(label="車主編號（選填，可從 CDP 資料帶入）", precision=0)
                load_btn = gr.Button("📥 從 CDP 帶入車主資料")
                cdp_info_box = gr.Textbox(label="CDP 車主資料", interactive=False, lines=4)
                name_input = gr.Textbox(label="您的稱呼", placeholder="例如：魏先生")
                car_input = gr.Dropdown(CAR_CHOICES, label="目前駕駛車款（可直接輸入其他車款）", value="RAV4",
                                        allow_custom_value=True)
                mileage_input = gr.Textbox(label="目前大約里程數", placeholder="例如：23000")
                km_input = gr.Slider(minimum=5000, maximum=50000, step=1000, value=15000,
                                     label="預估年行駛里程 (公里)")
                pref_input = gr.Textbox(label="您的用車習慣與喜好", placeholder="例如：每天通勤")
                confirm_btn = gr.Button("✅ 確認並建立檔案（同時精算）", variant="primary")
                calc_btn = gr.Button("⚡ 重新精算 5 年 TCO 養車成本")
                status_text = gr.Textbox(label="系統狀態", interactive=False)

            with gr.Column(scale=1):
                gr.Markdown("### 💰 5 年持有成本診斷")
                chart_output = gr.Plot(label="5 年持有成本比較圖")
                output_text = gr.Markdown("點擊「確認並建立檔案」或「重新精算」獲取診斷分析報告...")

        gr.Markdown("---")

        # 第二排：GenAI 對話顧問視窗
        gr.Markdown("### 💬 步驟 2：24H 購車與充電疑慮諮詢（GenAI 隨行顧問）")
        try:
            chatbot = gr.Chatbot(label="顧問回覆", height=450, type="messages")
        except TypeError:  # 新版 Gradio 已移除 type 參數，預設即 messages 格式
            chatbot = gr.Chatbot(label="顧問回覆", height=450)
        with gr.Row():
            msg_input = gr.Textbox(label="請輸入您的問題", placeholder="請先在上方點擊「確認並建立檔案」...",
                                   scale=4, interactive=False)
            submit_btn = gr.Button("📤 送出", scale=1)
        clear_btn = gr.ClearButton([msg_input, chatbot], value="🗑️ 清除對話")

        # 綁定按鈕事件 (Event Handlers)
        load_btn.click(load_owner, inputs=[owner_id_input], outputs=[status_text, cdp_info_box])
        dashboard_io = dict(fn=update_dashboard, inputs=[car_input, km_input], outputs=[chart_output, output_text])
        confirm_btn.click(confirm_profile,
                          inputs=[name_input, car_input, mileage_input, km_input, pref_input],
                          outputs=[status_text, msg_input]).then(**dashboard_io)
        calc_btn.click(**dashboard_io)

        chat_inputs = [msg_input, chatbot, name_input, car_input, mileage_input, km_input, pref_input, cdp_info_box]
        msg_input.submit(chat_with_ai, inputs=chat_inputs, outputs=[msg_input, chatbot])
        submit_btn.click(chat_with_ai, inputs=chat_inputs, outputs=[msg_input, chatbot])
    return demo


demo = build_app()
# Gradio 預設同一個函式一次只處理 1 個請求；放寬後多人同時提問才不會互相排隊
demo.queue(default_concurrency_limit=8)
# share=True 會生成一組外網可開啟的免費臨時連結 (xxxx.gradio.live)
demo.launch(share=True, debug=True)